### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp05'
DATA_URLS = {'greek_sites.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp05/greek_sites.csv'}
SCHEMA = {'greek_sites.csv': {'kind': 'csv', 'fields': ['site_id', 'name', 'longitude', 'latitude', 'category']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP05 — CRS Without Fear

Συμπλήρωσε τα κενά και κράτησε σημειώσεις για το τι αλλάζει όταν μετασχηματίζουμε CRS. Οι αριθμοί δεν αρκούν: πρέπει να ξέρεις αν είναι degrees ή metres και αν το CRS δηλώθηκε ή μετασχηματίστηκε.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path


In [ ]:
data_path = DATA_ROOT / "gp05/greek_sites.csv"
df = pd.read_csv(data_path)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs='EPSG:____')
gdf.head()


## Έλεγχος αρχικού CRS
Εκτύπωσε CRS, πλήθος γραμμών, τύπους γεωμετρίας και bounds.

In [ ]:
print('CRS:', gdf.crs)
print('rows:', len(gdf))
print('geometry types:', gdf.geom_type.value_counts().to_dict())
print('bounds:', gdf.total_bounds)


In [ ]:
# Μετασχημάτισε σε Greek Grid για εργασία σε μέτρα
projected = gdf.to_crs('EPSG:____')
projected[['name', 'geometry']]


## QA και σύγκριση bounds
Έλεγξε ότι δεν άλλαξαν οι γραμμές ή η γεωμετρία και εξήγησε γιατί άλλαξαν οι αριθμητικές τιμές των bounds.

Controlled wrong-CRS example: δημιούργησε προσωρινά `wrong = gdf.set_crs('EPSG:2100', allow_override=True)` και σύγκρινε bounds με το σωστό `projected`. Τι αποδεικνύει ότι οι αριθμοί έμειναν ίδιοι αλλά η ερμηνεία έγινε λάθος;

In [ ]:
assert len(projected) == len(gdf)
assert projected.crs is not None
assert projected.geom_type.eq('Point').all()
print('initial bounds:', gdf.total_bounds)
print('projected bounds:', projected.total_bounds)


In [ ]:
# Mini challenge: υπολόγισε την απόσταση Αθήνας–Θεσσαλονίκης σε μέτρα
# distance_m = ...
# print(distance_m / 1000)
